In [48]:
import nltk
import re
import os 
import pandas as pd
nltk.download('punkt')

from nltk.tokenize import sent_tokenize

CLAIM_ACTION_VERBS = {
    "achieved", "implemented", "reduced", "increased", "measured",
    "verified", "reported", "audited", "deployed", "executed",
    "tracked", "established", "completed", "delivered"
}

MEASUREMENT_TOKENS = {
    "baseline", "benchmark", "metric", "kpi", "indicator",
    "percent", "percentage", "pct", "million", "billion",
    "data", "quantified", "measured"
}

TARGET_VERBS = {
    "goal", "target", "commit", "aim", "reach", "attain",
    "plan", "intend", "will"
}



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [46]:
def split_sentences(text):
    return sent_tokenize(text)

In [60]:
def tokenize(text):
    return set(re.findall(r"\b[a-zA-Z]+\b", text.lower()))

def is_valid_claim(text):
    tokens = tokenize(text)

    has_action = any(v in tokens for v in CLAIM_ACTION_VERBS)
    has_target = any(v in tokens for v in TARGET_VERBS)
    has_measure = any(m in tokens for m in MEASUREMENT_TOKENS)
    has_number = bool(re.search(r"\d", text))

    # Valid if:
    # 1) Outcome (action + measure/number)
    # 2) Commitment (target + time/number)
    # 3) Accountable process (action + accountability verb)
    if has_action and (has_measure or has_number):
        return True

    if has_target and (has_number or "by" in tokens):
        return True

    return False


def classify_claim_scope(text):
    tokens = tokenize(text)

    has_action = any(v in tokens for v in CLAIM_ACTION_VERBS)
    has_target = any(v in tokens for v in TARGET_VERBS)
    has_measure = any(v in tokens for v in MEASUREMENT_TOKENS)

    if has_action and has_measure:
        return "Outcome"
    if has_target and not has_measure:
        return "Commitment"
    if has_action:
        return "Process"
    return "Other"

def classify_specificity(text):
    tokens = tokenize(text)

    score = 0
    score += sum(1 for t in tokens if t in MEASUREMENT_TOKENS)
    score += len(re.findall(r"\d+(\.\d+)?", text))  # numbers

    if score >= 3:
        return "Specific"
    if score == 2:
        return "Semi-Specific"
    return "Vague"



In [ ]:
def load_vocab_by_section(path):
    sections = {}
    current_section = None

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            # Skip empty lines and separators
            if not line or set(line) == {"="}:
                continue

            line_lower = line.lower()

            # Detect section headers
            if line_lower.startswith("environmental"):
                current_section = "ENVIRONMENTAL"
                sections[current_section] = set()
                continue

            if line_lower.startswith("social"):
                current_section = "SOCIAL"
                sections[current_section] = set()
                continue

            if line_lower.startswith("governance"):
                current_section = "GOVERNANCE"
                sections[current_section] = set()
                continue

            if line_lower.startswith("greenwashing"):
                current_section = "GREENWASHING"
                sections[current_section] = set()
                continue

            if line_lower.startswith("substantive"):
                current_section = "SUBSTANTIVE"
                sections[current_section] = set()
                continue

            # Parse keyword lines
            if current_section:
                words = [w.strip().lower() for w in line.split(",") if w.strip()]
                sections[current_section].update(words)

    return sections


VOCAB = load_vocab_by_section("expanded_esg_keywords.txt")
print(VOCAB)
ENV_WORDS = VOCAB["ENVIRONMENTAL"]
SOC_WORDS = VOCAB["SOCIAL"]
GOV_WORDS = VOCAB["GOVERNANCE"]
GREENWASH_WORDS = VOCAB["GREENWASHING"]
SUBSTANTIVE_WORDS = VOCAB["SUBSTANTIVE"]



{'ENVIRONMENTAL': {'irrigate', 'carbon neutral', 'carbon footprint', 'charge plate', 'climate change', 'decarbonization', 'ecological', 'wastefulness', 'greenhouse', 'greenish', 'solar', 'net zero', 'recycle', 'disforestation', 'conservation', 'nature', 'fossil', 'water supply', 'emanation', 'roll', 'ocean', 'sea', 'renewable', 'clean energy', 'fogy', 'william green', 'muscularity', 'biodiversity', 'malarkey', 'waste', 'pliant', 'deforestation', 'forest', 'department of energy', 'green', 'glasshouse', 'footprint', 'ghg emissions', 'global warming', 'preservation', 'defilement', 'environmental', 'environmental impact', 'carbon copy', 'contamination', 'clime', 'resource efficiency', 'climate', 'carbon', 'pollution', 'emission', 'ecosystem', 'sustainable', 'mood', 'timber', 'run off', 'bionomic', 'energy', 'circular economy', 'recycling', 'step', 'footmark', 'sustainability', 'carbon dioxide', 'water', 'wind', 'woodland', 'renewable energy', 'plastic', 'co2', 'atomic number 6'}, 'SOCIAL':

In [4]:
def classify_pillar(sentence):
    s = sentence.lower()
    if any(w in s for w in ENV_WORDS):
        return "Environmental"
    if any(w in s for w in SOC_WORDS):
        return "Social"
    if any(w in s for w in GOV_WORDS):
        return "Governance"
    return "Other"


In [52]:
def classify_claim_nature(sentence):
    s = sentence.lower()

    has_greenwash = any(w in s for w in GREENWASH_WORDS)
    has_substantive = any(w in s for w in SUBSTANTIVE_WORDS)

    if has_greenwash and not has_substantive:
        return "Aspirational"
    if has_substantive and not has_greenwash:
        return "Substantive"
    if has_greenwash and has_substantive:
        return "Mixed"
    return "Neutral"

def assess_greenwashing_risk(text, specificity, claim_scope):
    tokens = tokenize(text)

    greenwash_count = sum(1 for t in tokens if t in GREENWASH_WORDS)
    substantive_count = sum(1 for t in tokens if t in SUBSTANTIVE_WORDS)

    if specificity == "Vague" and greenwash_count > substantive_count:
        return "High"

    if claim_scope == "Commitment" and specificity != "Specific":
        return "Medium"

    return "Low"



In [5]:
def classify_claim_type(sentence):
    s = sentence.lower()
    if "by " in s or "target" in s:
        return "Target"
    if any(w in s for w in ["reduced", "achieved", "completed", "delivered"]):
        return "Performance"
    if any(w in s for w in ["policy", "committee", "framework"]):
        return "Governance"
    return "Process"


In [8]:
def classify_specificity(sentence):
    s = sentence.lower()

    if re.search(r"\b(by|within|across|from|through)\b", s):
        return "Semi-Specific"
    if len(s.split()) >= 20:
        return "Specific"
    return "Vague"


In [9]:
def classify_greenwashing_risk(sentence):
    nature = classify_claim_nature(sentence)
    specificity = classify_specificity(sentence)

    if nature == "Aspirational" and specificity == "Vague":
        return "High"
    if nature == "Aspirational":
        return "Medium"
    if nature == "Mixed":
        return "Medium"
    return "Low"


In [ ]:
CLAIM_ACTORS = ["we ", "our ", "the company", "ls co", "levi"]


def is_esg_sentence(sentence):
    tokens = re.findall(r"\b[a-zA-Z]+\b", sentence.lower())
    return (
        any(t in ENV_WORDS for t in tokens) or
        any(t in SOC_WORDS for t in tokens) or
        any(t in GOV_WORDS for t in tokens)
    )


In [61]:
def classify_claims_pipeline(clean_text, company_name):
    sentences = split_sentences(clean_text)

    # Step 1: ESG filtering
    esg_sentences = [s for s in sentences if is_esg_sentence(s)]

    records = []

    for sentence in esg_sentences:

        # Step 2: HARD claim gate (prevents headers & boilerplate)
        if not is_valid_claim(sentence):
            continue

        # Step 3: Core classifications
        pillar = classify_pillar(sentence)
        claim_scope = classify_claim_scope(sentence)
        specificity = classify_specificity(sentence)
        greenwashing_risk = assess_greenwashing_risk(
            sentence, specificity, claim_scope
        )

        records.append({
            "company": company_name,
            "claim_text": sentence,
            "pillar": pillar,
            "claim_scope": claim_scope,          # NEW
            "specificity": specificity,
            "greenwashing_risk": greenwashing_risk,
            "word_count": len(sentence.split())
        })

    return pd.DataFrame(records)


In [81]:
def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()
clean_text = load_txt("clean_text/Marathon_clean.txt")
claims_df = classify_claims_pipeline(
    clean_text,
    "Marathon"
)




In [82]:
for i, text in claims_df["claim_text"].items():
    print(f"\n--- Row {i} ---\n{text}")



--- Row 0 ---
We re in the Business of Accelerating Life s Possibilities  12.4 billion 348  2.8 billion The energy and products we produce support industrial and residential needs across a broad value chain, including: standard cubic feet per day vessels and barges gallons of renewable fuels of natural gas processing owned and operated delivered in 2024 capacity through marine business Fuel for cars, trucks, buses, Fuel for steam and electricity Fuel for outdoor heaters, grills, aircraft, trains and ships generation, heating and cooling cooktops and lanterns  21,000 559  40 million miles of pipeline owned, transport trucks owned barrels of terminal storage leased or with ownership and operated capacity interest Ingredients for other product manufacturing, such as plastics, fertilizers, synthetic fibers, cosmetics, pharmaceuticals,  8,900 2 strong  13,550 solvents and aerosol propellant brands North American retail and rail tank cars marketing locations Marathon  and ARCO  Asphalt for 

Dataset created: (456, 3)
